# 🧠 MindSpace: Custom Mental Health Chatbot Model Training
### **Final Year Project — Group No. 12**
**Savitribai Phule Shikshan Prasarak Mandal's**  
**N. B. Navale Sinhgad College of Engineering, Solapur**  
**Department of Computer Science & Engineering**  
**Under Guidance of:** Prof. L. C. Maindargi  
**Team Members:** Rajiv Kore, Saurabh Chopade, Adarsh Swadde, Aniket Bankar  

---
### 📌 Overview
This notebook fine-tunes an open-source conversational foundation model (**Microsoft DialoGPT-medium**, 345M parameters) on curated psychotherapy datasets (**CounselChat** + **Mental Health Counseling Conversations**) to produce a dedicated, self-hosted mental wellness AI for **MindSpace**.

## ⚡ Step 1: Verify Free GPU & Install Dependencies
Make sure your Colab runtime is set to **T4 GPU** (`Runtime -> Change runtime type -> T4 GPU`).

In [ ]:
!nvidia-smi
!pip install -q transformers datasets accelerate scikit-learn matplotlib tqdm torch

## 📚 Step 2: Download and Prepare Psychotherapy Datasets
We load and merge real therapist-answered counseling conversations from Hugging Face.

In [ ]:
import os
import math
import torch
from datasets import load_dataset
from torch.utils.data import Dataset
import matplotlib.pyplot as plt
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    Trainer,
    TrainingArguments,
    TrainerCallback
)

dialogues = []

# Dataset 1: fadodr/mental_health_therapy (8,580+ samples)
try:
    ds1 = load_dataset("fadodr/mental_health_therapy", split="train")
    for item in ds1:
        c = (item.get("instruction", "") or "").strip()
        r = (item.get("output", "") or "").strip()
        if len(c) > 10 and len(r) > 10:
            dialogues.append({"user": c, "assistant": r})
    print(f"Loaded {len(dialogues)} dialogues from fadodr/mental_health_therapy")
except Exception as e:
    print("Dataset 1 load error:", e)

# Dataset 2: alexandreteles/mental-health-conversational-data (660+ samples)
try:
    ds2 = load_dataset("alexandreteles/mental-health-conversational-data", split="train")
    count_ds2 = 0
    for item in ds2:
        q = (item.get("Context", "") or "").strip()
        a = (item.get("Response", "") or "").strip()
        if len(q) > 10 and len(a) > 10:
            dialogues.append({"user": q, "assistant": a})
            count_ds2 += 1
    print(f"Loaded {count_ds2} dialogues from alexandreteles dataset")
except Exception as e:
    print("Dataset 2 load error:", e)

print(f"Total Curated Training Dialogues: {len(dialogues)}")

## 🤖 Step 3: Initialize Tokenizer & Model (DialoGPT-medium)
We add special boundary tokens (`<|startoftext|>`, `<|sep|>`, `<|endoftext|>`) for clean conversational turns.

In [ ]:
BASE_MODEL = 'microsoft/DialoGPT-medium'
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

SPECIAL_TOKENS = {
    'pad_token': '<|pad|>',
    'bos_token': '<|startoftext|>',
    'eos_token': '<|endoftext|>',
    'sep_token': '<|sep|>'
}
tokenizer.add_special_tokens(SPECIAL_TOKENS)

model = AutoModelForCausalLM.from_pretrained(BASE_MODEL)
model.resize_token_embeddings(len(tokenizer))

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.to(device)
print(f'✅ Model initialized on {device.upper()}')

## 🔄 Step 4: Tokenize & Construct Dataset Class

In [ ]:
class CounselingDataset(Dataset):
    def __init__(self, data, tokenizer, max_length=256):
        self.examples = []
        for d in data:
            text = f"{tokenizer.bos_token}User: {d['user']}{tokenizer.sep_token}Counselor: {d['assistant']}{tokenizer.eos_token}"
            enc = tokenizer(
                text,
                truncation=True,
                max_length=max_length,
                padding='max_length',
                return_tensors='pt'
            )
            input_ids = enc['input_ids'].squeeze(0)
            attention_mask = enc['attention_mask'].squeeze(0)
            labels = input_ids.clone()
            labels[labels == tokenizer.pad_token_id] = -100
            self.examples.append({
                'input_ids': input_ids,
                'attention_mask': attention_mask,
                'labels': labels
            })

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        return self.examples[idx]

# 90% Train / 10% Validation Split
split = int(len(dialogues) * 0.9)
train_dataset = CounselingDataset(dialogues[:split], tokenizer)
val_dataset = CounselingDataset(dialogues[split:], tokenizer)
print(f'Train samples: {len(train_dataset)} | Val samples: {len(val_dataset)}')

## 🚀 Step 5: Train Model with AdamW & Cosine Learning Rate Schedule

In [ ]:
class LossCallback(TrainerCallback):
    def __init__(self):
        self.train_losses = []
        self.eval_losses = []
        self.steps = []
        self.eval_steps = []

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs:
            if 'loss' in logs:
                self.train_losses.append(logs['loss'])
                self.steps.append(state.global_step)
            if 'eval_loss' in logs:
                self.eval_losses.append(logs['eval_loss'])
                self.eval_steps.append(state.global_step)

training_args = TrainingArguments(
    output_dir='./mindspace_custom_model',
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=5e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    lr_scheduler_type='cosine',
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=1,
    logging_steps=25,
    fp16=torch.cuda.is_available(),
    report_to='none'
)

loss_tracker = LossCallback()
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    callbacks=[loss_tracker]
)

print('🚀 Starting training...')
trainer.train()
print('✅ Training finished successfully!')

## 📊 Step 6: Plot Loss Curve & Compute Perplexity
Screenshot this graph to show your project guide and add to your Presentation & Report!

In [ ]:
eval_res = trainer.evaluate()
ppl = math.exp(eval_res['eval_loss']) if eval_res['eval_loss'] < 30 else 999.9
print(f'🎯 Final Eval Loss: {eval_res["eval_loss"]:.4f}')
print(f'🎯 Model Perplexity (PPL): {ppl:.2f}')

# Plotting Loss Curve
plt.figure(figsize=(9, 5), dpi=200)
if loss_tracker.train_losses:
    plt.plot(loss_tracker.steps, loss_tracker.train_losses, label='Training Loss', color='#8C1D22', linewidth=2)
if loss_tracker.eval_losses:
    plt.plot(loss_tracker.eval_steps, loss_tracker.eval_losses, label='Validation Loss', color='#0D9488', linewidth=2, linestyle='--')

plt.title('MindSpace Chatbot Model: Cross-Entropy Loss over Training Steps', fontsize=12, fontweight='bold')
plt.xlabel('Training Steps', fontsize=10)
plt.ylabel('Loss', fontsize=10)
plt.grid(True, linestyle=':', alpha=0.6)
plt.legend()
plt.tight_layout()
plt.savefig('mindspace_loss_curve.png')
plt.show()

## 💬 Step 7: Test Your Trained Model with Emotional Prompts

In [ ]:
def chat_with_model(user_query, max_new_tokens=100):
    prompt = f"{tokenizer.bos_token}User: {user_query}{tokenizer.sep_token}Counselor:"
    inputs = tokenizer(prompt, return_tensors='pt').to(device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
            temperature=0.7,
            top_p=0.9,
            top_k=50,
            repetition_penalty=1.2,
            do_sample=True
        )
    response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    return response.strip()

test_queries = [
    "I am having severe anxiety about failing my engineering exams.",
    "I feel like I have nobody to talk to and I am completely overwhelmed.",
    "I feel so exhausted today and I cannot focus on anything."
]

for q in test_queries:
    print(f'\n👤 User: {q}')
    print(f'🤖 MindSpace AI: {chat_with_model(q)}')

## 📦 Step 8: Save & Download Trained Model Weights
Run this cell to save the model and download the zip file to put into your project.

In [ ]:
SAVE_PATH = './mindspace_custom_model'
trainer.save_model(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)
print(f'✅ Model saved to {SAVE_PATH}')

# Zip for easy 1-click download
!zip -r mindspace_custom_model.zip ./mindspace_custom_model
print('🎉 mindspace_custom_model.zip is ready to download from Colab Files panel!')